In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import scipy.stats as stats
from torch.ao.nn.quantized.functional import threshold
import seaborn as sns
from scaling.utils import functional_form_L0, fit_parametric_form, functional_form_L0_max, fit_linear_model
from itertools import product




In [ ]:
def get_pareto_frontier(
        df: pd.DataFrame,
        x_name="flops",
        y_name="Validation Loss",
        minimization=True,
        pareto=True
) -> pd.DataFrame:
    """ Function to compute Pareto over FLOPs.
    """
    # NOTE: strict assumption here that x_name is maximized and y_name is minimized
    df_copy = df.copy()
    if not minimization:
        df_copy[y_name] = - df_copy[y_name]

    if pareto:
        df_sorted = df_copy.sort_values(by=[x_name, y_name], ascending=[True, True])
        df_sorted = df_sorted.drop_duplicates(subset=[x_name], keep="first")
        pareto_points = []
        min_loss_so_far = float('inf')
        for _, row in df_sorted.iterrows():
            if row[y_name] < min_loss_so_far:
                pareto_points.append(row)
                min_loss_so_far = row[y_name]

        df_copy = pd.DataFrame(pareto_points)
    else:
        idx = df_copy.groupby(x_name)[y_name].idxmin()
        df_copy = df_copy.loc[idx].sort_values(by=x_name)

    if not minimization:
        df_copy[y_name] = - df_copy[y_name]
    return df_copy


---

In [ ]:
# here, define the suitable path for relative sourcing for the rest of the notebook
CWD = Path.cwd()
BASE_PATH = CWD.parent / "data" / "nanotab"
FIGURE_PATH = CWD / "figures" / "nanotabpfn" / "seeds"
FIGURE_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = CWD / "output" / "nanotabpfn"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

metrics = ["real_data/balanced_accuracy", "real_data/nll", "val/val_loss", "real_data/accuracy", "real_data/roc_auc"]
scaling_parameters = ["total_cells", "parameters", "total_flops"]
supporting_parameters = ["seed", "config_id"]

CWD

In [ ]:
# data read

PARENT_SEED = 42
SEEDS = [1826, 1926, 2126, 2226]
SEEDS.append(PARENT_SEED)

DATANAMES = (
    # "s1",
    "s1_seed=1826",
    "s1_seed=1926",
    "s1_seed=2126",
    "s1_seed=2226",
    # "s1.1",
    "s1.1_seed=1826",
    "s1.1_seed=1926",
    "s1.1_seed=2126",
    "s1.1_seed=2226",
    # "s1.2",
    "s1.2_seed=1826",
    "s1.2_seed=1926",
    "s1.2_seed=2126",
    "s1.2_seed=2226",
    # "s2",
    # "s2_seed=1826",
    # "s2_seed=1926",
    # "s2_seed=2126",
    # "s2_seed=2226",
)

dfs = []
for dataname in DATANAMES:
    _filename = f"run_summary_{dataname}.pickle.xz"
    df = pd.read_pickle(BASE_PATH / _filename)
    dfs.append(df)

In [ ]:
df = pd.concat(dfs, ignore_index=True)

# /work/dlclarge2/hogj-scaling_nano_tabpfn/results/scaling/s1.2_seed=1826/configs/config_156
import re

def extract_scale_and_config_id(path: str) -> str:
    match = re.search(
        r"/scaling/(s\d+(?:\.\d+)?)_seed=\d+/configs/config_(\d+)",
        path
    )

    if not match:
        raise ValueError(f"Could not extract scale and config_id from: {path}")

    scale = match.group(1)
    config_id = int(match.group(2))

    return f"{scale}-{config_id}"

df["config_id"] = df["config_path"].map(extract_scale_and_config_id)
print(df.head())
print(df.shape)
print(df.columns)

In [ ]:
# basic sanity check

for _seed in df.seed.unique():
    _df = df.loc[df.seed == _seed, "parameters"]
    print(_seed, _df.value_counts().to_dict())
    print()

# retaining only the expected seeds (TODO/NOTE: disable this if needed)

df = df.loc[df.seed.isin(SEEDS)]

In [ ]:
# handling hidden dim column naming conflict

df["config/mlp_hidden_multiple"] = df["config/mlp_hidden_multiple"].combine_first(df["config/model_config.mlp_hidden_multiple"])
df = df.drop(columns=["config/model_config.mlp_hidden_multiple"])

In [ ]:
# collecting main configs 
_hps = sorted([col for col in df.columns if col.startswith("config/")])

counts = df.groupby("seed")[_hps].nunique()
print(counts)

# drop columns that have same HP values across seeds (not useful for our analysis)
_cols_to_drop = counts.columns[(counts == 1).all()]
print("Dropping columns with same HP values across seeds:", _cols_to_drop.tolist())
df = df.loc[:, ~df.columns.isin(_cols_to_drop)]
print(df.head())

---

### NOTE: NEED TO DELETE SOME NANs AS FLOPs

In [ ]:


_df_floplist = df.loc[~df.total_flops.isna()].copy()
_df_noflops = df.loc[df.total_flops.isna()].copy()

# Some metrics were logged for flops and steps. We are removing the step logging.
# for metric in metrics:
#    _df_noflops[metric] = _df_noflops[metric].map(lambda x: x[(x.index > 1_000_000) & (x.index < 13000370000000)])
#    assert ((_df_noflops[metric].map(len)==9).all().all())

# _df_noflops["total_flops"] = _df_noflops[metrics[0]].map(lambda x: x.index)




# df = pd.concat([_df_floplist, _df_noflops]).copy()

# DOES NOT WORK. NEEDS TO BE RERUN?

In [ ]:
# Save the paths to the nan files for rerunning
with (OUTPUT_PATH / "bugged_seed_runs.txt").open("w", encoding="utf-8") as file:
    for item in _df_noflops["config_path"].to_list():
        file.write(f"{item}\n")

In [ ]:
_df_floplist = _df_floplist[(metrics + scaling_parameters + supporting_parameters)]

In [ ]:
every_step_columns = ["total_flops", 'total_cells']
leftover_columns = ["parameters", "seed", "config_id"]
df = None
for loss_column in metrics:
    next_df = pd.DataFrame(pd.concat(_df_floplist[loss_column].to_list(), keys=_df_floplist.index))
    if df is None:
        df = next_df
    else:
        df = df.join(next_df, how='outer', validate='one_to_one')

for every_step_column in every_step_columns:
    next_df = pd.DataFrame(pd.concat(_df_floplist[every_step_column].to_list(), keys=_df_floplist.index))
    df = df.join(next_df, how='left', validate='one_to_one')

df = df.join(_df_floplist[leftover_columns], how='left', on=df.index.get_level_values(0)).drop(
    columns="key_0")



In [ ]:
print(_df_floplist["real_data/nll"].shape)

In [ ]:
# FLOP ADJUSTMENT

import math
import numpy as np

FLOP_VALUES = np.array([
    1.00000000e+11, 1.77827941e+11, 3.16227766e+11, 5.62341325e+11,
    1.00000000e+12, 1.77827941e+12, 3.16227766e+12, 5.62341325e+12,
    1.00000000e+13, 1.77827941e+13, 3.16227766e+13, 5.62341325e+13,
    1.00000000e+14, 1.77827941e+14, 3.16227766e+14, 5.62341325e+14,
    1.00000000e+15, 1.77827941e+15, 3.16227766e+15, 5.62341325e+15,
    1.00000000e+16, 1.77827941e+16, 3.16227766e+16, 5.62341325e+16,
    1.00000000e+17
])


def substitute_closest(x, arr=FLOP_VALUES):
    idx = np.abs(arr - x).argmin()
    return arr[idx]


def round_sig(x, sig=2):
    if x == 0:
        return 0
    return round(x, sig - int(math.floor(math.log10(abs(x)))) - 1)

df['total_flops_rounded'] = df['total_flops'].map(substitute_closest)
df = df.drop(columns=["total_flops"])
# drop total_flops column as we have the rounded version now


In [ ]:
averaged_df = df.groupby(["config_id", "total_flops_rounded"]).agg({
    "parameters": "first",
    **{metric: "mean" for metric in metrics},
    "total_cells": "mean",
}).reset_index(drop=False)

In [ ]:

sns.set_theme(style="white")

summary_table_noise = []

for metric in metrics:
    # include the min max bullshit here
    if "accuracy" in metric or "roc_auc" in metric:
        min_value_df = df.loc[df.groupby(["seed", "total_flops_rounded"])[metric].idxmax()]
    else:
        min_value_df = df.loc[df.groupby(["seed", "total_flops_rounded"])[metric].idxmin()]

    seed_stats = (
        min_value_df.groupby("total_flops_rounded").agg(
            n_seeds=("seed", "nunique"),
            mean_loss=(metric, "mean"),
            seed_std=(metric, lambda x: x.std(ddof=1)),
            seed_sem=(metric, lambda x: x.std(ddof=1) / np.sqrt(x.count())),
            min_loss=(metric, "min"),
            max_loss=(metric, "max"),
        )
    )
    seed_stats[f"{metric}_seed_cv"] = seed_stats["seed_std"] / seed_stats["mean_loss"]
    seed_stats["range"] = seed_stats["max_loss"] - seed_stats["min_loss"]

    summary_table_noise.append(seed_stats[[f"{metric}_seed_cv"]])

    fig, ax = plt.subplots(1, 1, figsize=(6, 4))

    ax.errorbar(
        seed_stats.index,
        seed_stats["mean_loss"],
        yerr=seed_stats["seed_std"],
        fmt="o",
        capsize=4,
    )
    ax.set_xscale("log")
    ax.set_xlabel("FLOPs")
    ax.set_ylabel(metric)
    fig.suptitle("Mean loss ± seed std across scales")
    fig.savefig(FIGURE_PATH / f"{metric}_seed_variance.pdf".replace("/", "_"))

    plt.show()



    fig, ax = plt.subplots(1, 1, figsize=(6, 4))
    sns.scatterplot(
        data=min_value_df,
        x="total_flops_rounded",
        y=metric,
        # hue="seed",
        ax=ax
    )

    ax.set_xscale("log")
    ax.set_xlabel("FLOPs")
    ax.set_ylabel(metric)
    save_path = FIGURE_PATH / "seed_variance" / f"{metric}_seed_variance.pdf".replace("/", "_")
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path)
    plt.show()

    # what do I wanna do with this?
    # Just plot the metrics first
    # and their uncertainty across seeds
    # then

# Look at the variance of the seed values

# I could also plot this is a table

# There is a difference in the average best and the best average....

# let me plot it anyways
summary_table_noise = pd.concat(summary_table_noise, axis=1)
latex_table = summary_table_noise.to_latex(index=True)

with (FIGURE_PATH / "seed_variance" / "relative_std.txt").open("w", encoding="utf-8") as f:
    f.write(latex_table)
print(summary_table_noise)

In [ ]:
sns.set_theme(style="white")

param_names = {0: "L0", 1: "a", 2: "alpha"}

initial_grid_min = [
    [0, 0.1, 0.5],
    [0.1, 1, 2, 5, 10, 50, 100],
    [-1, -0.5, 0, 0.5, 1]
]
bounds_min = [
    (1e-6, 1),
    (0, 100),
    (-5, 5)
]

initial_grid_max = [
    [0, 1, 2, 4, 8],
    [-30, -20, -10, -5, -2.5, -1, -0.5, 0],
    [-2, -1, -0.5, -0.25, 0, 0.25, 0.5, 1, 2]
]
bounds_max = [
    (0, 10),
    (-40, 0),
    (-5, 5)
]

In [ ]:
# Fit all the scaling laws
return_pareto = True

flops_column = "total_flops_rounded"

seed_fits = []
for metric in metrics:
    fig, ax = plt.subplots(layout='constrained')
    maximization = "roc_auc" in metric or "accuracy" in metric
    minimization = not maximization


    if minimization:
        initial_grid = initial_grid_min
        bounds = None
        func_form = functional_form_L0
    else:
        initial_grid = initial_grid_max
        bounds = None
        func_form = functional_form_L0_max

    pareto = get_pareto_frontier(averaged_df, flops_column, metric, minimization=minimization, pareto=return_pareto)

    ax.scatter(pareto[flops_column], pareto[metric], color="black")

    best_params, best_loss = fit_parametric_form(
        func_form=func_form,
        X_data=pareto[flops_column].values,
        y_data=np.log(pareto[metric].values),
        initial_grid=list(product(*initial_grid)),
        bounds=bounds,
        delta=1e-3,
        use_scipy=True,
    )

    y = func_form(FLOP_VALUES, best_params, return_log_loss=False)
    ax.plot(
        FLOP_VALUES,
        y,
        linestyle="--",
        color="black",
    )

    avrg_fit_metric = {
        param_names[0]: best_params[0],
        param_names[1]: best_params[1],
        param_names[2]: best_params[2],
        f"predicted_loss_at_{FLOP_VALUES[-1]:.1e}": y[-1]
    }
    seed_fits_metric = []

    for seed in df["seed"].unique():
        setting_df = df[df["seed"]==seed].copy()

        pareto = get_pareto_frontier(setting_df, flops_column, metric, minimization=minimization, pareto=return_pareto)

        ax.scatter(pareto[flops_column], pareto[metric], alpha=0.3)

        best_params, best_loss = fit_parametric_form(
            func_form=func_form,
            X_data=pareto[flops_column].values,
            y_data=np.log(pareto[metric].values),
            initial_grid=list(product(*initial_grid)),
            bounds=bounds,
            delta=1e-3,
            use_scipy=True,
        )
        y = func_form(FLOP_VALUES, best_params, return_log_loss=False)
        ax.plot(
            FLOP_VALUES,
            y,
            linestyle="--",
            alpha=0.3,
        )

        seed_fits_metric.append({
            param_names[0]: best_params[0],
            param_names[1]: best_params[1],
            param_names[2]: best_params[2],
            f"predicted_loss_at_{FLOP_VALUES[-1]:.1e}": y[-1]
        })

    ax.set_xscale("log")
    ax.set_xlabel("FLOPs")
    ax.set_ylabel(metric)
    if minimization:
        ax.set_yscale("log")

    save_path = FIGURE_PATH / "mean_vs_individual_fit" / f"{metric}.pdf".replace("/", "_")
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path)

    seed_fits_metric = pd.DataFrame(seed_fits_metric)
    for column in seed_fits_metric.columns:
        seed_fits.append({
            "metric": metric.replace("_", " ").split("/")[-1],
            "quantity": column.replace("_", " ").replace("/", " "),
            "avrg fit": avrg_fit_metric[column],
            "mean": seed_fits_metric[column].mean(),
            "std": seed_fits_metric[column].std(),
            "rel std pct": abs(seed_fits_metric[column].std() / seed_fits_metric[column].mean()) * 100,
            "min": seed_fits_metric[column].min(),
            "max": seed_fits_metric[column].max(),
        })


seed_fits = pd.DataFrame(seed_fits)
seed_fits = seed_fits.set_index(["metric", "quantity"])

latex_table = seed_fits.to_latex(index=True, float_format="%.2f")

with (FIGURE_PATH / "mean_vs_individual_fit" / "latex_table.txt").open("w", encoding="utf-8") as f:
    f.write(latex_table)

print(seed_fits.head())


In [ ]:
column

---

### Looking at Irreducible Loss sensitivity to seed + loss-scaling fit variance

In [ ]:
plt.clf()

METRIC = "val/val_loss"

_colors = ["red", "blue", "green", "black", "purple", "cyan", "magenta", "yellow"]

for j, _seed in enumerate(_df_floplist.seed.unique()):
    _df = _df_floplist.loc[_df_floplist.seed == _seed]
    for i in range(len(_df)):
        row = _df.iloc[i]
        plt.scatter(row.total_flops_rounded, row[METRIC], color=_colors[j], alpha=0.5)

    plt.xscale("log")
    plt.xlabel("Total FLOPs (rounded)")
    plt.ylabel(METRIC)

In [ ]:
plt.clf()

METRIC = "real_data/accuracy"

_colors = ["red", "blue", "green", "black", "purple", "cyan", "magenta", "yellow"]

for j, _seed in enumerate(_df_floplist.seed.unique()):
    _df = _df_floplist.loc[_df_floplist.seed == _seed]
    for i in range(len(_df)):
        row = _df.iloc[i]
        plt.scatter(row.total_flops_rounded, row[METRIC], color=_colors[j], alpha=0.5)

    plt.xscale("log")
    plt.xlabel("Total FLOPs (rounded)")
    plt.ylabel(METRIC)

In [ ]:
plt.clf()

METRIC = "real_data/nll"

_colors = ["red", "blue", "green", "black", "purple", "cyan", "magenta", "yellow"]

for j, _seed in enumerate(_df_floplist.seed.unique()):
    _df = _df_floplist.loc[_df_floplist.seed == _seed]
    for i in range(len(_df)):
        row = _df.iloc[i]
        plt.scatter(row.total_flops_rounded, row[METRIC], color=_colors[j], alpha=0.5)

    plt.xscale("log")
    plt.xlabel("Total FLOPs (rounded)")
    plt.ylabel(METRIC)

In [ ]:
plt.clf()

METRIC = "real_data/roc_auc"

_colors = ["red", "blue", "green", "black", "purple", "cyan", "magenta", "yellow"]

for j, _seed in enumerate(_df_floplist.seed.unique()):
    _df = _df_floplist.loc[_df_floplist.seed == _seed]
    for i in range(len(_df)):
        row = _df.iloc[i]
        plt.scatter(row.total_flops_rounded, row[METRIC], color=_colors[j], alpha=0.5)

    plt.xscale("log")
    plt.xlabel("Total FLOPs (rounded)")
    plt.ylabel(METRIC)

In [ ]:
# pareto per seed
# fit L0, alpha, A
# compare variance across seeds

In [ ]:
# working dataframe

# _df_floplist = df  # NOTE: use this if data preprocessing consistent

display(_df_floplist.head())

##### Per SEED


In [ ]:
# FITTING L0 FORM WITH PARETO PER SEED

_METRIC = "val/val_loss"
param_names = {0: "L0", 1: "a", 2: "alpha"}

initial_grid = [
    [0, 0.1, 0.5],
    [0.1, 1, 2, 5, 10, 50, 100],
    [-1, -0.5, 0, 0.5, 1]
]
bounds = [
    (1e-6, 1),
    (0, 100),
    (-5, 5)
]

_SEEDS = list(set(SEEDS) - set([PARENT_SEED]))
n_seeds = len(_SEEDS)
fig, axes = plt.subplots(1, n_seeds, figsize=(8 * n_seeds, 6), sharey=True)
if n_seeds == 1:
    axes = [axes]

all_params = []

for ax, seed in zip(axes, sorted(_SEEDS)):
    # subset for seed
    _df = _df_floplist.loc[_df_floplist.seed == seed]

    # flatten dataframe for seed
    df_flat = _df.explode(["total_flops_rounded", _METRIC]).reset_index(drop=True)

    _pareto = get_pareto_frontier(
        df_flat,
        x_name="total_flops_rounded",
        y_name=_METRIC,
    )

    ax.scatter(df_flat.total_flops_rounded, df_flat[_METRIC], alpha=0.5)
    ax.scatter(_pareto.total_flops_rounded, _pareto[_METRIC], color="red")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(f"seed={seed}", fontsize=20)
    ax.set_xlabel("FLOPs", fontsize=20)
    ax.tick_params(axis="both", labelsize=20)

    best_params, best_loss = fit_parametric_form(
        func_form=functional_form_L0,
        X_data=_pareto.total_flops_rounded.values,
        y_data=_pareto[_METRIC].values,
        initial_grid=list(product(*initial_grid)),
        bounds=bounds,
        delta=1e-3,
        use_scipy=True,
    )

    ax.plot(
        FLOP_VALUES,
        functional_form_L0(FLOP_VALUES, best_params),
        linestyle="--",
        color="black",
    )

    all_params.append({"seed": seed, **{param_names[i]: v for i, v in enumerate(best_params)}})

params_df = pd.DataFrame(all_params).set_index("seed")

axes[0].set_ylabel(_METRIC, fontsize=20)
axes[0].set_ylim(0.6, 1.5)
plt.tight_layout()
plt.show()

# Summary table
summary = pd.concat([
    params_df.mean().rename("mean"),
    params_df.std().rename("std."),
], axis=1)
print(params_df.to_string())
print("\nSummary:")
print(summary.to_string())


##### All SEEDs


In [ ]:
# FITTING L0 FORM WITH PARETO FOR ALL SEEDS

_METRIC = "val/val_loss"
param_names = {0: "L0", 1: "a", 2: "alpha"}

initial_grid = [
    [0, 0.1, 0.5],
    [0.1, 1, 2, 5, 10, 50, 100],
    [-1, -0.5, 0, 0.5, 1]
]
bounds = [
    (1e-6, 1),
    (0, 100),
    (-5, 5)
]

_SEEDS = list(set(SEEDS) - set([PARENT_SEED]))
n_seeds = len(_SEEDS)
fig, ax = plt.subplots(1, 1, figsize=(4, 3))

all_params = []

_df = _df_floplist.loc[_df_floplist.seed.isin(_SEEDS)]
# flatten dataframe for seed
df_flat = _df.explode(["total_flops_rounded", _METRIC]).reset_index(drop=True)

_pareto = get_pareto_frontier(
    df_flat,
    x_name="total_flops_rounded",
    y_name=_METRIC,
)

ax.scatter(df_flat.total_flops_rounded, df_flat[_METRIC], alpha=0.5)
ax.scatter(_pareto.total_flops_rounded, _pareto[_METRIC], color="red")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title(f"seed={seed}", fontsize=20)
ax.set_xlabel("FLOPs", fontsize=20)
ax.tick_params(axis="both", labelsize=20)

best_params, best_loss = fit_parametric_form(
    func_form=functional_form_L0,
    X_data=_pareto.total_flops_rounded.values,
    y_data=_pareto[_METRIC].values,
    initial_grid=list(product(*initial_grid)),
    bounds=bounds,
    delta=1e-3,
    use_scipy=True,
)

ax.plot(
    FLOP_VALUES,
    functional_form_L0(FLOP_VALUES, best_params),
    linestyle="--",
    color="black",
)

all_params.append({"seed": seed, **{param_names[i]: v for i, v in enumerate(best_params)}})

params_df = pd.DataFrame(all_params).set_index("seed")

ax.set_ylabel(_METRIC, fontsize=20)
# ax.set_ylim(0.6, 1.5)
plt.tight_layout()
plt.show()

# Summary table
summary = pd.concat([
    params_df.mean().rename("mean"),
    params_df.std().rename("std."),
], axis=1)
print(params_df.to_string())
# print("\nSummary:")
# print(summary.to_string())


---

### Looking at config membership to Non-dominated fronts

In [ ]:
_retain_cols = _hps + ["parameters", "seed", "total_flops_rounded", _METRIC]

_df = _df_floplist.loc[_df_floplist.seed == _seed]
_pareto = get_pareto_frontier(
    # flattened _df
    _df.explode(["total_flops_rounded", _METRIC]).reset_index(drop=True),
    x_name="total_flops_rounded",
    y_name=_METRIC,
)
_pareto = _pareto[[c for c in _retain_cols if c in _pareto.columns]]

In [ ]:
# Generating Pareto/per seed and taking union of all such configs to see how they appear across seeds

_retain_cols = _hps + ["parameters", "seed", "total_flops_rounded", _METRIC]

config_matrix = {}

for _seed in set(SEEDS) - set([PARENT_SEED]):
    _df = _df_floplist.loc[_df_floplist.seed == _seed]
    _pareto = get_pareto_frontier(
        # flattened _df
        _df.explode(["total_flops_rounded", _METRIC]).reset_index(drop=True),
        x_name="total_flops_rounded",
        y_name=_METRIC,
    )
    _pareto = _pareto[[c for c in _retain_cols if c in _pareto.columns]]
    # unique configs in this seed's pareto
    pareto_configs = set(_pareto[_hps].apply(tuple, axis=1))
    config_matrix[_seed] = pareto_configs

# union of all configs as index
all_configs = sorted(set().union(*config_matrix.values()))

# creates a dataframe with unique HP configs as index
df_config = pd.DataFrame(
    {seed: [1 if cfg in configs else 0 for cfg in all_configs]
     for seed, configs in config_matrix.items()},
    index=pd.MultiIndex.from_tuples(all_configs, names=_hps),
)

# column holding how many times the unique HP features in a Pareto across seeds
df_config["membership"] = df_config.sum(axis=1)
df_config.sort_values("membership", ascending=False, inplace=True)
display(df_config)

In [ ]:
# Generating Pareto fronts iteratively and assigning front numbers to configs for each seed

_retain_cols = _hps + ["parameters", "seed", "total_flops_rounded", _METRIC]
K = 5  # number of fronts to compute

config_matrix = {}

for _seed in set(SEEDS) - {PARENT_SEED}:
    _df = _df_floplist.loc[_df_floplist.seed == _seed]
    df_flat = _df.explode(["total_flops_rounded", _METRIC]).reset_index(drop=True)

    remaining = df_flat.copy()
    seed_config_fronts = {}  # config_tuple -> front number

    for front in range(1, K + 1):
        if remaining.empty:
            break
        _pareto = get_pareto_frontier(
            remaining,
            x_name="total_flops_rounded",
            y_name=_METRIC,
        )
        _pareto = _pareto[[c for c in _retain_cols if c in _pareto.columns]]

        for cfg in _pareto[_hps].apply(tuple, axis=1):
            if cfg not in seed_config_fronts:  # first front wins
                seed_config_fronts[cfg] = front

        # remove pareto points from remaining
        pareto_idx = _pareto.index
        remaining = remaining.drop(index=pareto_idx, errors="ignore")

    config_matrix[_seed] = seed_config_fronts

# union of all configs as index
all_configs = sorted(set().union(*[set(d.keys()) for d in config_matrix.values()]))

df_config = pd.DataFrame(
    {seed: [d.get(cfg, None) for cfg in all_configs]
     for seed, d in config_matrix.items()},
    index=pd.MultiIndex.from_tuples(all_configs, names=_hps),
)

df_config["min_front"] = df_config.min(axis=1)
df_config["max_front"] = df_config.max(axis=1)
df_config["front_range"] = df_config["max_front"] - df_config["min_front"]
df_config["no_fronts"] = df_config.isna().sum(axis=1)

# df_config.sort_values("front_range", ascending=False, inplace=True)
df_config.sort_values("no_fronts", ascending=True, inplace=True)
display(df_config)


In [ ]:
# CALCULATING SPEARMAN RANK CORRELATION OF FRONT MEMBERSHIP ACROSS ALL SEEDS

seed_cols = list(set(SEEDS) - set([PARENT_SEED]))

result = stats.spearmanr(df_config[seed_cols], nan_policy="omit")

corr_matrix = pd.DataFrame(result.statistic, index=seed_cols, columns=seed_cols)
pval_matrix = pd.DataFrame(result.pvalue, index=seed_cols, columns=seed_cols)
display(corr_matrix)
display(pval_matrix)

In [ ]:
# Score each config by its mean front rank across seeds, penalized by inconsistency.
# NaNs (not in any front) are treated as front 2*K — worse than the last observed front.
# Lower score = consistently appears in earlier fronts across seeds.
filled = df_config[seed_cols].fillna(2 * K)
df_config["mean_std_score"] = filled.mean(axis=1) + filled.std(axis=1)

# Fraction of seeds where this config appeared in the top `top_k` fronts.
# Combined with std of front ranks to penalize inconsistency.
# Final score: lower top_front_rate and higher variance both push score up (worse).
top_k = 2
df_config["top_front_score"] = df_config[seed_cols].apply(
    lambda row: (row <= top_k).sum() / len(seed_cols), axis=1
)
df_config["top_front_score"] = -df_config["top_front_score"] + df_config[seed_cols].fillna(2 * K).std(axis=1)


df_config.sort_values(["top_front_score", "mean_std_score", "min_front"], inplace=True)
display(df_config)